# Judge the Judge — MVP

Workshop harness for evaluating LLM-as-judge prompts on [LLMBar](https://github.com/princeton-nlp/LLMBar)
(Zeng et al., ICLR 2024): instruction/response pairs with gold labels for which
response objectively follows the instruction, where the *worse* response of an
adversarial pair is deliberately more superficially appealing.

**How it works:** you write a judge prompt; the harness runs it over a sample of
pairs — every pair in **both presentation orders** — and scores your judge against
the gold labels. The harness is identical in every round: only the prompt changes,
so score changes are attributable to the prompt. A full run costs well under a
cent with `gpt-4.1-nano`.


## 1. Setup — run and move on

In [ ]:
%pip install -q openai pandas matplotlib
import json, os, random, re, subprocess, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

MODEL = "gpt-4.1-nano"   # if round 2 shows no lift, try "gpt-4.1-mini" — prompting
                         # gains concentrate in stronger judges (LLMBar paper, Table 3)
N_PER_SUBSET = 8         # pairs per LLMBar subset (5 subsets -> 40 pairs, 80 calls/run)
SEED = 42                # fixed so everyone in the room judges the same pairs
MAX_WORKERS = 5          # modest concurrency so 20 people on one key don't trip rate limits

In [ ]:
# API key: Colab secret -> environment variable -> paste prompt
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENAI_API_KEY")
    except Exception:
        pass
if not api_key:
    from getpass import getpass
    api_key = getpass("Paste the workshop OpenAI API key: ")

from openai import OpenAI
client = OpenAI(api_key=api_key)

## 2. Load the evaluation pairs

In [ ]:
if not Path("LLMBar").exists():
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/princeton-nlp/LLMBar.git"], check=True)

SUBSETS = ["Natural", "Neighbor", "GPTInst", "GPTOut", "Manual"]
REQUIRED_KEYS = {"input", "output_1", "output_2", "label"}


def load_subset(name):
    # Locate the subset's json inside the clone without hard-coding the layout.
    candidates = [p for p in Path("LLMBar").rglob("*.json")
                  if p.parent.name == name and "Dataset" in p.parts]
    for path in sorted(candidates):
        items = json.loads(path.read_text())
        if isinstance(items, list) and items and REQUIRED_KEYS <= set(items[0]):
            return [{"subset": name, "pair_id": f"{name}-{i}",
                     "instruction": it["input"], "output_1": it["output_1"],
                     "output_2": it["output_2"], "gold": int(it["label"])}
                    for i, it in enumerate(items)]
    raise FileNotFoundError(f"could not find data for subset {name!r}")


rng = random.Random(SEED)
pairs = []
for name in SUBSETS:
    subset = load_subset(name)
    pairs += rng.sample(subset, min(N_PER_SUBSET, len(subset)))

print(f"{len(pairs)} pairs sampled:",
      pd.Series([p["subset"] for p in pairs]).value_counts().to_dict())

In [ ]:
# Eyeball one adversarial pair so you know what the judge is up against
ex = next(p for p in pairs if p["subset"] == "Neighbor")
print("INSTRUCTION:\n", ex["instruction"])
print("\nRESPONSE 1:\n", ex["output_1"][:600])
print("\nRESPONSE 2:\n", ex["output_2"][:600])
print("\nGOLD:", ex["gold"])

## 3. The harness — run these two cells and move on

Every pair is judged twice (original order and swapped), at temperature 0 with a
fixed seed, so results are as repeatable as the API allows and position bias is
measured for free. You don't need to read this code to play.

In [ ]:
def parse_verdict(text):
    """Last standalone 1 or 2 in the judge's reply, or None if unparseable."""
    found = re.findall(r"\b([12])\b", text)
    return int(found[-1]) if found else None


def call_judge(prompt_template, pair, flipped, retries=5):
    r1, r2 = pair["output_1"], pair["output_2"]
    if flipped:
        r1, r2 = r2, r1
    prompt = prompt_template.format(
        instruction=pair["instruction"], response_1=r1, response_2=r2)
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=0,   # modal verdict — sampling noise would blur prompt comparisons
                seed=SEED,       # best-effort determinism (not guaranteed by OpenAI)
                messages=[{"role": "user", "content": prompt}])
            break
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt + random.random())
    text = resp.choices[0].message.content or ""
    pick_position = parse_verdict(text)   # what the judge saw on screen
    # map back to the underlying response number
    pick = None if pick_position is None else (3 - pick_position if flipped else pick_position)
    return {"pair_id": pair["pair_id"], "subset": pair["subset"], "flipped": flipped,
            "pick_position": pick_position, "pick": pick,
            "correct": None if pick is None else pick == pair["gold"],
            "prompt_tokens": resp.usage.prompt_tokens,
            "completion_tokens": resp.usage.completion_tokens,
            "raw": text}


def run_judge(prompt_template, pairs):
    jobs = [(p, flipped) for p in pairs for flipped in (False, True)]
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        records = list(ex.map(lambda j: call_judge(prompt_template, *j), jobs))
    return pd.DataFrame(records)

In [ ]:
PRICE_IN, PRICE_OUT = 0.10, 0.40   # $ per 1M tokens, gpt-4.1-nano


def summarize(df):
    correct = df["correct"].astype("boolean")
    by_order = df.pivot(index="pair_id", columns="flipped", values="pick")
    consistent = by_order[False] == by_order[True]
    consistent_ids = consistent[consistent].index
    # An inconsistent pair = the judge stuck to the same on-screen *position*
    # across the swap; this measures which position it stuck to.
    sticky = df[~df.pair_id.isin(consistent_ids)]
    first_share = (sticky["pick_position"] == 1).mean() if len(sticky) else float("nan")
    return {
        "accuracy": correct.mean(),
        "accuracy_natural": correct[df.subset == "Natural"].mean(),
        "accuracy_adversarial": correct[df.subset != "Natural"].mean(),
        "positional_consistency": consistent.mean(),
        "accuracy_on_consistent_pairs": correct[df.pair_id.isin(consistent_ids)].mean(),
        "first_position_share_when_inconsistent": first_share,
        "parse_failures": int(df["pick"].isna().sum()),
        "cost_usd": (df.prompt_tokens.sum() * PRICE_IN
                     + df.completion_tokens.sum() * PRICE_OUT) / 1e6,
    }


def subset_accuracy(df):
    return (df.assign(c=df["correct"].astype("boolean"))
              .groupby("subset")["c"].mean().reindex(SUBSETS))


def group_stats(df):
    """Natural vs Adversarial: mean pair-level accuracy + 95% CI half-width."""
    d = df.assign(c=df["correct"].astype("boolean").astype("Float64"),
                  group=df.subset.where(df.subset == "Natural", "Adversarial"))
    per_pair = d.groupby(["group", "pair_id"])["c"].mean()
    out = per_pair.groupby("group").agg(["mean", "sem", "count"])
    out["ci"] = 1.96 * out["sem"]
    return out.reindex(["Natural", "Adversarial"])


RUNS = {}


def evaluate(name, prompt_template):
    df = run_judge(prompt_template, pairs)
    RUNS[name] = df
    stats = summarize(df)
    return pd.Series({k: (round(float(v), 3) if isinstance(v, float) else v)
                      for k, v in stats.items()}, name=name)

## 4. Write your judge prompt

Rules of the game:

- Use the placeholders `{instruction}`, `{response_1}`, `{response_2}` — the
  harness fills them in (and handles the order-swapping for you).
- Your prompt must make the model **end its reply with the number of the better
  response: `1` or `2`** — the harness parses the last standalone 1/2 in the reply.

In [ ]:
MY_PROMPT = """You are comparing two responses to an instruction.

Instruction:
{instruction}

Response 1:
{response_1}

Response 2:
{response_2}

Which response is better? Reply with only the number 1 or 2."""

## 5. Round 1 — run your judge

What the numbers mean:

- **accuracy** — agreement with the gold labels, over all calls (both orders)
- **accuracy_natural / accuracy_adversarial** — normal pairs vs trap pairs; the
  *gap* between them is your judge's susceptibility to surface appeal
- **positional_consistency** — share of pairs with the same verdict in both
  orders; the rest were decided by position, not content
- **accuracy_on_consistent_pairs** — quality of the verdicts you could actually
  trust (a real pipeline keeps these and re-adjudicates the rest)
- **first_position_share_when_inconsistent** — of the position-decided pairs, how
  often position 1 won (≈0.5 means no directional lean; the direction varies by
  model — small models often lean toward the *last* response they read)

In [ ]:
evaluate("round1", MY_PROMPT)

## 6. Results

Natural vs Adversarial accuracy, with 95% confidence intervals — at this sample
size the intervals are wide, which is worth internalizing before reading too much
into any single number.

In [ ]:
import matplotlib.pyplot as plt

SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#898781", "#e1e0d9"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # fixed slot order, never cycled
GROUPS = ["Natural", "Adversarial"]


def plot_runs(names=None):
    names = (names or list(RUNS))[:4]   # >4 series stops being readable — facet instead
    fig, ax = plt.subplots(figsize=(7, 4), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    group_w = 0.7
    width = group_w / len(names)
    for i, name in enumerate(names):
        gs = group_stats(RUNS[name])
        xs = [j - group_w / 2 + (i + 0.5) * width for j in range(len(GROUPS))]
        ax.bar(xs, gs["mean"], width * 0.9, color=SERIES[i], label=name, zorder=3)
        ax.errorbar(xs, gs["mean"], yerr=gs["ci"], fmt="none",
                    ecolor=MUTED, elinewidth=1, capsize=3, zorder=4)
        for x, v in zip(xs, gs["mean"]):
            if pd.notna(v):
                ax.text(x, 0.04, f"{v:.0%}", ha="center", fontsize=9,
                        color="#ffffff", zorder=5)
    ax.axhline(0.5, color=MUTED, linewidth=1, linestyle=(0, (4, 3)), zorder=2)
    ax.text(len(GROUPS) - 0.58, 0.515, "chance", color=MUTED, fontsize=8)
    ax.set_xticks(range(len(GROUPS)),
                  ["Natural\n(normal pairs)", "Adversarial\n(trap pairs)"], color=MUTED)
    ax.set_ylim(0, 1.08)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0],
                  ["0%", "25%", "50%", "75%", "100%"], color=MUTED)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(color=GRID, labelcolor=MUTED)
    if len(names) > 1:
        ax.legend(frameon=False, loc="upper right", labelcolor=INK)
    ax.set_title("Judge accuracy (95% CI)", color=INK, loc="left")
    plt.tight_layout()
    plt.show()


plot_runs()

In [ ]:
# Read the judge's own words on the pairs it got wrong — often the most
# convincing exhibit: watch it praise the polished answer that ignores the task.
wrong = RUNS["round1"].query("correct == False")
for _, row in wrong.head(2).iterrows():
    p = next(p for p in pairs if p["pair_id"] == row.pair_id)
    print("INSTRUCTION:", p["instruction"][:200])
    print("JUDGE SAID:", row.raw[:200])
    print("(gold was", p["gold"], "— judge picked", row.pick, ")\n")

In [ ]:
# Optional drill-down: accuracy per LLMBar subset. The four adversarial subsets
# differ in how the trap was constructed (see the paper); at this sample size
# each column is noisy — read directions, not decimals.
pd.concat({name: subset_accuracy(df) for name, df in RUNS.items()}, axis=1).round(2)

## 7. Round 2 — apply the theory, judge again

Rewrite your prompt using what the research says actually works (LLMBar paper,
Table 3): explicit **rules** that prioritize instruction-following over style,
self-generated **metrics**, or a self-generated **reference** answer. Two traps
the literature warns about while you experiment:

- Plain "think step by step" tends **not** to help — free-form reasoning drifts
  toward the prettier answer and rationalizes it. Structured criteria do better.
- Telling the judge to "ignore the order of the responses" rarely fixes position
  bias — try it and watch `positional_consistency`. The reliable fix is
  pipeline-level: judge both orders and only trust consistent verdicts (which the
  harness already measures as `accuracy_on_consistent_pairs`).

Also note: prompting gains concentrate in stronger judges. If nothing moves the
adversarial number, the lesson may be that your judge *model* is too weak — try
`MODEL = "gpt-4.1-mini"` (setup cell) and rerun.

In [ ]:
RULES_PROMPT = """You are comparing two responses to an instruction. Select the \
response that better follows the instruction.

Evaluation rules:
1. First identify precisely what the instruction asks for, including any \
constraints or required content.
2. A response that does exactly what was asked beats one that is longer, \
friendlier, or better formatted but misses or ignores any part of the instruction.
3. Do not reward extra information that was not requested.
4. Instruction compliance and accuracy outweigh style, tone, and length.

Instruction:
{instruction}

Response 1:
{response_1}

Response 2:
{response_2}

In one sentence, state what the instruction requires. Then end your reply with \
the number of the better response: 1 or 2."""

evaluate("round2", RULES_PROMPT)

In [ ]:
plot_runs()

## Next steps (not in this MVP)

- **Model check**: rerun both rounds with `gpt-4.1-mini` — pick the cheapest
  judge that shows a clear naive→informed lift
- **Verbosity metric**: padded variants of wrong answers at increasing lengths —
  dose-response curve of accuracy vs length ratio
- **Self-preference metric**: flawed answers restyled by the judge's own model vs
  another model, same content
- **Shared leaderboard**: final cell POSTs each participant's metrics to a Google
  Sheet via an Apps Script webhook